# Práctica OCR: Extracción de texto

En este notebook practicaremos tres enfoques fundamentales de OCR (Optical Character Recognition):
1. **OCR Clásico con Tesseract**: Aprenderemos la importancia del preprocesado.
2. **OCR Moderno con PaddleOCR**: Uso de modelos robustos para escenas reales.
3. **Pipeline YOLO + OCR**: El flujo completo de detección y reconocimiento.
4. **Postprocesado**: Uso de Regex para limpiar datos.

---

In [ ]:
!sudo apt-get install tesseract-ocr tesseract-ocr-spa -y
!pip install pytesseract "paddleocr<3" "paddlepaddle<3" opencv-python ultralytics

In [ ]:
import cv2
import pytesseract
import matplotlib.pyplot as plt
import numpy as np
from paddleocr import PaddleOCR
from ultralytics import YOLO
from google.colab.patches import cv2_imshow
from IPython.display import clear_output
import requests
import re
from PIL import Image
from io import BytesIO
from yt_dlp import YoutubeDL

def download_img(url):
    response = requests.get(url)
    img = Image.open(BytesIO(response.content))
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

def show_img(img, title="Imagen"):
    plt.figure(figsize=(10,10))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

## Preprocesado en OCR

Los motores de OCR (especialmente los clásicos como Tesseract) funcionan mejor cuando el texto es **negro puro sobre fondo blanco puro**, sin ruido ni sombras.

### Técnicas:
1. **Escala de Grises**: Simplifica la imagen eliminando el color.
2. **Denoising (Filtros)**: Elimina el "granulado" o manchas del papel.
3. **Thresholding (Umbralizado)**: Convierte la imagen a blanco y negro (binaria).

### Preprocesado común:
```python
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
# Filtro para suavizar y quitar ruido
blur = cv2.GaussianBlur(gray, (5, 5), 0)
# Binarización de Otsu (calcula el umbral automáticamente)
_, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
```

El preprocesado a usar depende de la imagen, hay que pensar:
 - ¿Me ayudará usar escala de grises?
 - ¿El texto es fino o tiene ruido para suavizarlo?
 - ¿El fondo es uniforme o complejo?

# Ejercicio 1: Tesseract y preprocesado

Intentaremos leer un DNI. Verás que sin preprocesado los resultados pueden ser desastrosos.

## Qué hay que hacer:
1. Descargar la imagen del DNI.
2. Intentar pasar Tesseract directamente y observar el resultado.
3. Diseña y prueba 2 preprocesados distintos, por ejemplo:
    - Escala de grises
    - Escala de grises + blur
    - Escala de grises + threshold    
4. Prueba distintos valores de `--psm` según cómo creas que es la imagen:
    - 6  (si hay muchos bloques de texto)
    - 7  (si hay muchas líneas de texto)
    - 8  (si hay palabras sueltas)
    - 11 (si hay texto muy disperso)
5. Comprueba qué configuraciones obtienen mejor el nombre y el número de DNI

In [ ]:
#Imagen del DNI
IMG_URL = "https://raw.githubusercontent.com/jorgecs/apuntes/main/docs/ut5_ia_aplicada/4_yolo/img/dni.jpg" 
img_dni = download_img(IMG_URL)
show_img(img_dni, "DNI Original")

In [ ]:
#TODO: OCR directo


#TODO: Preprocesado (Grayscale -> Blur -> Threshold ...)

#TODO: Muestra la imagen preprocesada

#TODO: OCR tras preprocesado

# Ejercicio 2: PaddleOCR - OCR en escenas reales

PaddleOCR es mucho más potente para detectar texto inclinado o con fondos complejos.

## Qué hay que hacer:
1. Inicializar el modelo de PaddleOCR (idioma español).
2. Descargar una imagen de un cartel en la calle.
3. Procesar la imagen con PaddleOCR.
4. Extraer los resultados e imprimir: texto y confianza.
5. Dibuja sobre la imagen los cuadros detectados.

In [ ]:
#TODO: Inicializa PaddleOCR

#TODO: Busca una imagen de un cartel
SCENE_URL = ""
img_scene = download_img(SCENE_URL)

show_img(img_scene, "Imagen de Escena")

In [ ]:
#TODO: Procesa la imagen y visualiza resultados

# Ejercicio 3: Pipeline YOLO + OCR

Flujo en aplicaciones profesionales: Detectar -> Recortar -> Reconocer.

## Qué hay que hacer:
1. Cargar un modelo YOLO (`yolov8n.pt`).
2. Detectar un coche y su matrícula.
3. **Recorte**: Recorta el área de la matrícula.
4. **OCR**: Pasa el recorte por PaddleOCR o Tesseract.
5. **Resultado**: Muestra el recorte y el texto extraído.

In [ ]:
CAR_URL = "https://img.remediosdigitales.com/97800c/matricula-p-espana/1366_2000.jpg"
img_car = download_img(CAR_URL)

# TODO: Detectar matrícula

# TODO: Recorta y aplica OCR

# Ejercicio 4: Postprocesado (OCR + Regex)

Usaremos Regex para extraer solo los datos que nos importan de una cadena de texto "sucia".

## Qué hay que hacer:
1. Toma el texto sucio de un OCR.
2. Usa `re.findall` para extraer todos los precios (formato 0.00).
3. Identifica el precio más alto de la lista.

In [ ]:
texto_sucio = """
Factura #1234
Producto A ... 12.50 EUR
Producto B ... 5.99 EUR
IVA ... 2.10
TOTAL PAGADO ... 20.59
Gracias por su compra
"""

# TODO: Extrae los precios usando Regex

---